# EMG Gesture Classification — NinaPro DB1 (Exercise A)

**Goal:** classify 12 finger/thumb gestures (flexion/extension of index, middle, ring,
little finger; thumb adduction/abduction/flexion/extension) from 10-channel surface EMG,
as a first step toward EMG-driven prosthetic hand control.

**Dataset:** [NinaPro DB1](http://ninapro.hevs.ch), Subject 1, Exercise A
(Atzori et al., *Building the NinaPro Database*, BioRob 2012). Signals are pre-processed
by the database authors (rectified, low-pass filtered, resampled to 100 Hz).

**Pipeline overview:**
1. Load raw `.mat` data and explore the signal
2. Detect gesture repetition boundaries from the `restimulus` label
3. **Baseline v0** — one feature vector per repetition (naive, and why it fails)
4. **Final approach** — sliding-window feature extraction (200 ms windows, 50% overlap)
5. Classical ML: Random Forest, KNN
6. Neural network: MLP, with and without EarlyStopping
7. Results summary, limitations, and next steps

*Note on structure: earlier iterations of this notebook also tried a richer
single-vector-per-repetition feature set and a fixed 3-way sub-window split before
arriving at the sliding-window approach below. Those intermediate attempts are
summarized in Section 3 rather than kept as separate code, since the final sliding-window
method supersedes both and produces far more training data.*


## 1. Setup

In [ ]:
# scipy.io: for loading .mat files (MATLAB format)
import scipy.io as sio

# numpy: numerical operations (arrays, math)
import numpy as np

# matplotlib.pyplot: plotting EMG signals and training curves
import matplotlib.pyplot as plt
import seaborn as sns


## 2. Load and Inspect the Raw Data

In [ ]:
file_path = '../ninaprodb/s1/S1_A1_E1.mat'
data = sio.loadmat(file_path)

print("Keys in the .mat file:", data.keys())


Each `.mat` file stores several parallel signals, all synchronized sample-by-sample:

| Field | Meaning |
|---|---|
| `emg` | (n_samples, 10) — the 10-electrode sEMG signal, pre-processed to 100 Hz |
| `stimulus` | Raw movement label as originally presented to the subject |
| `restimulus` | **Corrected** movement label, realigned to when the subject actually moved (used throughout this project — see note below) |
| `repetition` | Which of the 10 repetitions of each gesture this sample belongs to |
| `subject`, `exercise` | Metadata identifying the recording |

**Why `restimulus` and not `stimulus`:** the raw `stimulus` signal marks *when the movie
cue appeared on screen*, but subjects react with a short delay. `restimulus` is
realigned to the subject's actual EMG onset, giving cleaner class boundaries. This
project uses `restimulus` throughout after an earlier version mistakenly used `stimulus`.

In [ ]:
emg = data['emg']
restimulus = data['restimulus'].flatten()
stimulus = data['stimulus'].flatten()
repetition = data['repetition'].flatten()
fs = 100  # Hz — NinaPro DB1 is pre-processed/resampled to 100 Hz by the database authors

print("EMG shape:", emg.shape)                # (n_samples, 10 electrodes)
print("Restimulus shape:", restimulus.shape)
print("Sampling frequency:", fs, "Hz")
print("Unique gestures:", np.unique(restimulus))   # 0 = rest, 1-12 = gestures
print("Unique repetitions:", np.unique(repetition))


## 3. Signal Exploration (EDA)

A quick sanity-check plot of two different gestures confirms the raw EMG looks reasonable and visually distinct before building any pipeline on top of it.

In [ ]:
def plot_gesture_raw(gesture_id, gesture_name, n_samples=200):
    idx = np.where(restimulus == gesture_id)[0]
    start, end = idx[0], idx[0] + n_samples
    plt.figure(figsize=(12, 5))
    for i in range(10):
        plt.plot(emg[start:end, i], label=f'Electrode {i+1}')
    plt.xlabel('Samples (at 100 Hz)')
    plt.ylabel('EMG Amplitude')
    plt.title(f'Preprocessed EMG — Gesture {gesture_id} ({gesture_name})')
    plt.legend(fontsize=8, ncol=2)
    plt.grid(alpha=0.3)
    plt.show()

plot_gesture_raw(1, "Index Flexion")
plot_gesture_raw(11, "Thumb Flexion")


**Finding repetition boundaries.** `restimulus` is a per-sample label; to build one
example per repetition (or per window) we need the *indices* where the label changes.
This is done by looking for jumps in `restimulus`.

In [ ]:
gesture_starts = np.where(np.diff(restimulus) != 0)[0] + 1  # +1: diff() shifts index left
gesture_starts = np.insert(gesture_starts, 0, 0)              # include sample 0

print("Number of label changes detected:", len(gesture_starts))
print("First 10 start indices:", gesture_starts[:10])
print("Gesture label at those starts:", restimulus[gesture_starts[:10]])


**Sanity-checking the sampling rate.** The protocol calls for ~5 s of movement and
3 s of rest per repetition. An early version of this notebook assumed `fs = 2000 Hz`
(a common raw NinaPro rate) and got timing that didn't match the protocol at all —
that mismatch is what revealed this particular `.mat` file is already resampled to
100 Hz, matching the paper's stated preprocessing.

In [ ]:
time_between_starts = np.diff(gesture_starts) / fs
print("Time between first 20 label changes (seconds), at fs=100Hz:")
print(np.round(time_between_starts[:20], 2))
print(f"\nMean: {time_between_starts.mean():.2f}s | Min: {time_between_starts.min():.2f}s | Max: {time_between_starts.max():.2f}s")


**Repetitions and durations per gesture** — confirms each of the 12 gestures has 10 repetitions, as expected from the NinaPro protocol.

In [ ]:
def get_repetition_windows(gesture_id):
    """Return (start, end) sample indices for each repetition of a gesture."""
    starts = gesture_starts[restimulus[gesture_starts] == gesture_id]
    windows = []
    for start in starts:
        next_idx = np.searchsorted(gesture_starts, start, side='right')
        end = gesture_starts[next_idx] if next_idx < len(gesture_starts) else len(restimulus)
        windows.append((start, end))
    return windows

for g in range(1, 13):
    windows = get_repetition_windows(g)
    durations = np.array([e - s for s, e in windows])
    print(f"Gesture {g:2d}: {len(windows)} reps, avg duration = {durations.mean():.1f} samples "
          f"({durations.mean()/fs:.2f}s)")


**Comparing gestures visually.** Averaging each repetition's EMG (trimmed to the shortest repetition length) and overlaying gestures on one electrode gives a first qualitative sense of separability.

In [ ]:
def mean_waveform(gesture_id, electrode=None):
    windows = get_repetition_windows(gesture_id)
    segments = [emg[s:e, :] for s, e in windows]
    min_len = min(seg.shape[0] for seg in segments)
    segments = [seg[:min_len, :] for seg in segments]
    mean_wave = np.mean(segments, axis=0)  # (min_len, 10)
    return mean_wave if electrode is None else mean_wave[:, electrode]

plt.figure(figsize=(12, 5))
plt.plot(mean_waveform(1, electrode=0), label="Gesture 1 (Index Flexion)", linewidth=2)
plt.plot(mean_waveform(2, electrode=0), label="Gesture 2 (Index Extension)", linewidth=2)
plt.xlabel("Samples (time)")
plt.ylabel("EMG Amplitude (Electrode 1)")
plt.title("Gesture 1 vs. Gesture 2 — Mean Waveform")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


In [ ]:
colors = plt.cm.tab20.colors
plt.figure(figsize=(14, 7))
for g in range(1, 13):
    plt.plot(mean_waveform(g, electrode=0), label=f"Gesture {g}", color=colors[g-1], linewidth=2)
plt.xlabel("Samples (time)")
plt.ylabel("EMG Amplitude (Electrode 1)")
plt.title("Mean EMG for All 12 Gestures (Electrode 1)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Baseline v0 — Naive One-Vector-Per-Repetition

**First attempt:** collapse each entire repetition down to a single feature vector
(mean EMG per electrode) and train directly on that. This is the simplest possible
approach and a useful baseline, but it has a fundamental data-volume problem:
12 gestures x 10 repetitions = only **120 samples total**. An 80/20 split leaves
just ~24 test samples, so per-class accuracy can only take a handful of discrete
values (0%, 50%, 100% for a 2-sample class) — nowhere near enough to trust as a
reliable metric.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_v0, y_v0 = [], []
for g in range(1, 13):
    for start, end in get_repetition_windows(g):
        X_v0.append(np.mean(emg[start:end, :], axis=0))  # mean per electrode -> 10 features
        y_v0.append(g)

X_v0, y_v0 = np.array(X_v0), np.array(y_v0)
print("X shape:", X_v0.shape, "  (12 gestures x 10 reps, 10 features)")

X_train_v0, X_test_v0, y_train_v0, y_test_v0 = train_test_split(
    X_v0, y_v0, test_size=0.2, random_state=42, stratify=y_v0
)
clf_v0 = RandomForestClassifier(n_estimators=100, random_state=42)
clf_v0.fit(X_train_v0, y_train_v0)
y_pred_v0 = clf_v0.predict(X_test_v0)

print(f"\nTest set size: {len(y_test_v0)} samples for 12 classes")
print(classification_report(y_test_v0, y_pred_v0, zero_division=0))


**Diagnosis:** the classification report above is noisy and unreliable purely due to
sample size — not a modeling problem. Two things were tried next before landing on the
real fix:

- **Richer features per repetition** (mean, std, min, max, RMS per electrode -> 50
  features instead of 10). This improved the *feature quality* per example but didn't
  address the core issue: still only 120 examples total.
- **Splitting each repetition into 3 fixed sub-windows** (start/middle/end thirds),
  tripling the dataset to 360 examples. Better, but still coarse and arbitrary.

**The actual fix:** slide a fixed-size window across each repetition with overlap,
generating many more (and more consistent) training examples per repetition. That's
the approach used from here on.


## 5. Final Feature Extraction — Sliding Window

Each repetition is scanned with a 200 ms window (20 samples at 100 Hz) and 50% overlap
(10-sample step). For every window, 5 statistics (mean, std, min, max, RMS) are computed
per electrode, giving 10 x 5 = **50 features per window**. Each window also records which
repetition it came from — needed for a leak-free train/test split (Section 6).

In [ ]:
window_size = 20   # 200ms at 100Hz
step_size = 10      # 100ms step -> 50% overlap

X, y, rep = [], [], []

for g in range(1, 13):
    for start, end in get_repetition_windows(g):
        rep_number = repetition[start]
        window_start = start
        while window_start + window_size <= end:
            window = emg[window_start: window_start + window_size, :]  # (20, 10)
            features = []
            for electrode in range(10):
                signal = window[:, electrode]
                features.extend([
                    np.mean(signal), np.std(signal), np.min(signal),
                    np.max(signal), np.sqrt(np.mean(signal ** 2)),
                ])
            X.append(features)
            y.append(g)
            rep.append(rep_number)
            window_start += step_size

X, y, rep = np.array(X), np.array(y), np.array(rep)

print("X shape:", X.shape, " (windows, features)")
print("Windows per gesture:")
for g in np.unique(y):
    print(f"  Gesture {g:2d}: {np.sum(y == g)} windows")
print("Repetitions present:", np.unique(rep))


## 6. Train/Test Split, Scaling, and Classical ML

**Repetition-based split (not random):** EMG samples within the same repetition are
temporally correlated (adjacent windows overlap by 50% and share muscle-activation
dynamics). A random split would leak information between train and test — the model
could partly "recognize" a repetition it has already partially seen rather than
generalizing to unseen movement instances. Splitting by whole repetitions (some
repetitions entirely held out) avoids this leakage and gives an honest accuracy estimate.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

test_reps = [2, 5, 7, 10]
train_mask = ~np.isin(rep, test_reps)
test_mask = np.isin(rep, test_reps)

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print("Train windows:", X_train.shape[0])
print("Test windows:", X_test.shape[0])

# Scale using train statistics only (avoid leaking test distribution into scaling)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)
print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, zero_division=0))

# --- KNN ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)
print("=== KNN ===")
print(classification_report(y_test, y_pred_knn, zero_division=0))

# --- Confusion matrix (Random Forest) ---
plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Blues',
            xticklabels=range(1, 13), yticklabels=range(1, 13))
plt.title("Confusion Matrix — Random Forest (sliding window, repetition split)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


## 7. Gesture 9 vs. 11 Confusion — Why It Happens

The confusion matrix above consistently shows gestures **9 (thumb adduction)** and
**11 (thumb flexion)** as the most confused pair. Both are thumb-only movements driven
by overlapping intrinsic/extrinsic thumb muscles, so some surface-EMG overlap is
anatomically expected rather than a modeling failure.


In [ ]:
def mean_waveform_pair(g1, g2):
    w1 = get_repetition_windows(g1)
    w2 = get_repetition_windows(g2)
    segs1 = [emg[s:e, :] for s, e in w1]
    segs2 = [emg[s:e, :] for s, e in w2]
    min_len = min(min(s.shape[0] for s in segs1), min(s.shape[0] for s in segs2))
    segs1 = [s[:min_len, :] for s in segs1]
    segs2 = [s[:min_len, :] for s in segs2]
    return np.mean(segs1, axis=0), np.mean(segs2, axis=0)

mean_g9, mean_g11 = mean_waveform_pair(9, 11)

fig, axes = plt.subplots(2, 5, figsize=(20, 8), sharex=True, sharey=True)
axes = axes.flatten()
for electrode in range(10):
    ax = axes[electrode]
    ax.plot(mean_g9[:, electrode], label="Gesture 9 (Thumb adduction)", linewidth=2)
    ax.plot(mean_g11[:, electrode], label="Gesture 11 (Thumb flexion)", linewidth=2)
    ax.set_title(f"Electrode {electrode + 1}")
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=8)
fig.supxlabel("Samples (time)")
fig.supylabel("EMG Amplitude")
fig.suptitle("Gesture 9 vs. Gesture 11 — All Electrodes", fontsize=14)
plt.tight_layout()
plt.show()


**Analysis:** across most electrodes the mean waveform shape is highly similar between
the two gestures, differing mainly in amplitude rather than pattern. **Electrode 8**
shows the clearest separation (sharper difference in peak timing/shape), making it the
most informative electrode for distinguishing this specific pair. Electrodes 4–6 show
almost no signal for either gesture, consistent with their placement being farther from
thumb-controlling muscles.

**Takeaway:** this confusion is a partly genuine sensor-placement limitation, not purely
a modeling bug — a fair limitation to note in the final write-up rather than something
to "fix" by tuning hyperparameters.

**Feature importance**, aggregated across the 5 statistics per electrode, cross-checks this: electrodes closer to the more expressive muscles should score higher.

In [ ]:
feature_names = [f"E{e+1}_{stat}" for e in range(10) for stat in ["mean", "std", "min", "max", "rms"]]
importances = rf.feature_importances_

# Aggregate the 5 stats per electrode into one importance score per electrode
electrode_importance = importances.reshape(10, 5).sum(axis=1)

plt.figure(figsize=(10, 5))
plt.bar(range(1, 11), electrode_importance)
plt.xticks(range(1, 11))
plt.xlabel("Electrode")
plt.ylabel("Summed feature importance (RF)")
plt.title("Random Forest Feature Importance by Electrode")
plt.grid(axis='y', alpha=0.3)
plt.show()


## 8. Neural Network (MLP)

A small multilayer perceptron on the same 50-dimensional feature vectors, as a point of
comparison against the tree-based models.

**Lesson learned (documented honestly):** an earlier run of this model used
`validation_split` on **class-ordered, unshuffled** data. Since Keras's `validation_split`
just takes the *last* N% of the array — not a random sample — entire gesture classes
were silently excluded from validation. The fix is a joint shuffle of `X_train`/`y_train`
right before fitting, shown below.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils import shuffle

# Keras' sparse_categorical_crossentropy expects 0-indexed labels
y_train_nn = y_train - 1
y_test_nn = y_test - 1

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(12, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:
# Shuffle jointly before fitting -- critical, see note above
X_train_shuf, y_train_shuf = shuffle(X_train_scaled, y_train_nn, random_state=42)

history = model.fit(
    X_train_shuf, y_train_shuf,
    validation_split=0.15,
    epochs=50,
    batch_size=32,
    verbose=1,
)


In [ ]:
y_pred_nn = model.predict(X_test_scaled).argmax(axis=1) + 1  # back to 1-12 labels

print("=== Neural Network (MLP) ===")
print(classification_report(y_test, y_pred_nn, zero_division=0))

plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('MLP Training Curve')
plt.grid(alpha=0.3)
plt.show()


## 9. MLP + EarlyStopping

**Framing this honestly:** EarlyStopping is added here for **training reliability and
reproducibility**, not as an accuracy booster. In testing, it matched (rather than
beat) a well-chosen fixed epoch count. What it *does* provide: no need to manually guess
the right number of epochs, and automatic protection against overfitting if the
model/data change later (e.g. after adding a CNN or more subjects).

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Fresh model -- trained from scratch for a fair comparison against the MLP above
model_es = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(12, activation='softmax'),
])
model_es.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

history_es = model_es.fit(
    X_train_shuf, y_train_shuf,
    validation_split=0.15,
    epochs=100,          # raised -- EarlyStopping will cut it short
    batch_size=32,
    callbacks=[early_stop],
    verbose=1,
)


In [ ]:
y_pred_es = model_es.predict(X_test_scaled).argmax(axis=1) + 1

print("=== Neural Network (MLP + EarlyStopping) ===")
print(classification_report(y_test, y_pred_es, zero_division=0))

plt.plot(history_es.history['accuracy'], label='train accuracy')
plt.plot(history_es.history['val_accuracy'], label='val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('MLP + EarlyStopping -- Training Curve')
plt.grid(alpha=0.3)
plt.show()


## 10. Results Summary & Discussion

| Model | Test Accuracy | Notes |
|---|---|---|
| Random Forest | ~80% | Best classical baseline; robust to feature scale |
| KNN (k=5) | ~76% | Simple, sensitive to feature scaling |
| MLP | ~81–82% | Best overall; needed the shuffle fix to validate correctly |
| MLP + EarlyStopping | ~81–82% | Matches manually-tuned MLP; adds training robustness, not accuracy |

*(Exact numbers depend on the current run — rerun cells above to regenerate against these
placeholders once satisfied with this structure.)*

**Known limitation:** gestures 9 (thumb adduction) and 11 (thumb flexion) are
consistently the most confused pair. Section 7 shows this is a genuine sensor-placement
limitation (overlapping thumb muscle activity), not purely a modeling shortfall —
electrode 8 provides the most separation for this pair, while electrodes 4–6 carry
almost no discriminative signal for the thumb overall.

**Next steps:**
1. **CNN on raw windowed signals** (rather than hand-crafted features) — let the network
   learn its own temporal/spatial filters and compare against the RF/MLP baselines above.
2. **Animated hand visualization** driven by live predictions, to complete the
   "prosthetic hand simulation" framing from the project README.
3. Package this notebook + results + limitations into the portfolio README.
